In [72]:
import pandas as pd
import random
import re

# read the data
data = pd.read_csv("../../data/answerList_data.csv")
print(data.head())


   Answer.ID FailingMethod  Question.ID  Answer.duration  Answer.confidence  \
0          1      HIT02_24           15           72.061                  3   
1          2      HIT02_24           15          131.485                  5   
2          3      HIT02_24           15          129.046                  3   
3          4      HIT02_24           15          276.233                  3   
4          5      HIT02_24           15          268.145                  5   

   Answer.difficulty  TP  TN  FN  FP  ... Code.complexity  \
0                  4   0   0   0   1  ...               4   
1                  1   0   0   0   1  ...               4   
2                  4   0   0   0   1  ...               4   
3                  4   0   0   0   1  ...               4   
4                  4   0   1   0   0  ...               4   

                      Worker.ID Worker.score       Worker.profession  \
0                   3AI7C4g38-9            3                Hobbyist   
1             

In [73]:
# we need to be able to predict the accuracy of the answer
# an answer is accurate if the answer is correct (TP) and no false positives (FP) are present
data['accuracy'] = ((data['TP'] == 1) & (data['FP'] == 0)).astype(int)

In [76]:
# print accuracy tp tn fp fn columns
print(data[['accuracy', 'TP', 'TN', 'FP', 'FN']].head())

   accuracy  TP  TN  FP  FN
0         0   0   0   1   0
1         0   0   0   1   0
2         0   0   0   1   0
3         0   0   0   1   0
4         0   0   1   0   0


In [77]:
# Calculate Type-Token Ratio (TTR)
def calculate_ttr(text):
    # Tokenize the text by splitting on non-alphabetic characters and converting to lowercase
    text = str(text)
    tokens = re.findall(r'\b\w+\b', text.lower())  # Tokenize and ignore case
    types = set(tokens)  # Unique words
    num_tokens = len(tokens)
    num_types = len(types)
    
    # Calculate TTR
    return num_types / num_tokens if num_tokens > 0 else 0

# Apply the function to the 'Answer.explanation' column
data['TTR'] = data['Answer.explanation'].apply(calculate_ttr)

print(data['TTR'].head())
print(data['Answer.explanation'].head())

0    1.000000
1    0.750000
2    0.692308
3    0.896552
4    0.763636
Name: TTR, dtype: float64
0                     The color is black and not RGB. 
1    Color.black out of expected range; should be C...
2    The acceptable range of colors are red; green;...
3    It appears there's an issue with this method b...
4    The Color constructor is invoked just as it sh...
Name: Answer.explanation, dtype: object


In [78]:
# prepare the relevant columns

# Worker profession is a string, convert to a number
data["Worker.profession"] = data["Worker.profession"].astype('category').cat.codes

# participant score and profession, duration, explanation size/complexity, confidence and difficulty
relevant_columns=["Worker.score", "Worker.profession", "Answer.duration", "TTR", "Answer.confidence", "Answer.difficulty", "accuracy", "FailingMethod"]

# discard the rest
data = data[relevant_columns]

In [79]:
# Ensure reproducibility
random.seed(42)

# Get bug reports
failing_methods = data['FailingMethod'].unique()

# Create holdout and training set
number_of_holdout_bug_reports = 2
selected_failing_methods = random.sample(list(failing_methods), number_of_holdout_bug_reports)
holdout_set = data[data['FailingMethod'].isin(selected_failing_methods)]
training_set = data[~data['FailingMethod'].isin(selected_failing_methods)]



In [80]:
print(training_set['FailingMethod'].unique())
print(holdout_set['FailingMethod'].unique())

['HIT03_6' 'HIT04_7' 'HIT07_33' 'HIT08_54' 'HIT05_35' 'HIT06_51']
['HIT02_24' 'HIT01_8']


In [ ]:
# train a decision tree based approach e.g. random forest

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.metrics import precision_score, recall_score, classification_report, accuracy_score

# Use accuracy as the target
X = training_set.drop(columns=["accuracy", "FailingMethod"])
y = training_set["accuracy"]

# Split the data into training and validation sets for cross-validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

rf_classifier = RandomForestClassifier(random_state=42, n_estimators=100)

cv_predictions = cross_val_predict(rf_classifier, X, y, cv=5) # cross validation

# Calculate precision and recall
cv_precision = precision_score(y, cv_predictions, average='binary')
cv_recall = recall_score(y, cv_predictions, average='binary')

print("Cross-Validation Precision:", cv_precision)
print("Cross-Validation Recall:", cv_recall)

# Calculate precision and recall per bug report 
bug_report_metrics = {}
for bug_report in training_set["FailingMethod"].unique():
    mask = training_set["FailingMethod"] == bug_report
    # Filter both y and cv_predictions for the specific bug report
    report_precision = precision_score(y[mask], cv_predictions[mask], average='binary')
    report_recall = recall_score(y[mask], cv_predictions[mask], average='binary')
    bug_report_metrics[bug_report] = {"precision": report_precision, "recall": report_recall}

print("\nCross-Validation Metrics per Bug Report:")
for bug_report, metrics in bug_report_metrics.items():
    print(f"{bug_report}: Precision={metrics['precision']:.2f}, Recall={metrics['recall']:.2f}")

# Train the model on the training set
rf_classifier.fit(X_train, y_train)

Cross-Validation Precision: 0.0
Cross-Validation Recall: 0.0

Cross-Validation Metrics per Bug Report:
HIT03_6: Precision=0.00, Recall=0.00
HIT04_7: Precision=0.00, Recall=0.00
HIT07_33: Precision=0.00, Recall=0.00
HIT08_54: Precision=0.00, Recall=0.00
HIT05_35: Precision=0.00, Recall=0.00
HIT06_51: Precision=0.00, Recall=0.00


c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics

RandomForestClassifier(random_state=42)

In [ ]:
# predict on the holdout set
X_holdout = holdout_set.drop(columns=["accuracy", "FailingMethod"])
y_holdout = holdout_set["accuracy"]
holdout_predictions = rf_classifier.predict(X_holdout)

# Calculate precision and recall on the holdout set
holdout_precision = precision_score(y_holdout, holdout_predictions, average='binary')
holdout_recall = recall_score(y_holdout, holdout_predictions, average='binary')

print("\nHoldout Precision:", holdout_precision)
print("Holdout Recall:", holdout_recall)

# Calculate holdout set metrics per bug report
holdout_bug_report_metrics = {}
for bug_report in holdout_set["FailingMethod"].unique():
    mask = holdout_set["FailingMethod"] == bug_report
    report_precision = precision_score(y_holdout[mask], holdout_predictions[mask], average='binary')
    report_recall = recall_score(y_holdout[mask], holdout_predictions[mask], average='binary')
    holdout_bug_report_metrics[bug_report] = {"precision": report_precision, "recall": report_recall}

print("\nHoldout Set Metrics per Bug Report:")
for bug_report, metrics in holdout_bug_report_metrics.items():
    print(f"{bug_report}: Precision={metrics['precision']:.2f}, Recall={metrics['recall']:.2f}")


Holdout Precision: 0.0
Holdout Recall: 0.0

Holdout Set Metrics per Bug Report:
HIT02_24: Precision=0.00, Recall=0.00
HIT01_8: Precision=0.00, Recall=0.00


c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\bzwad\anaconda3\envs\DSWebscrape\lib\site-packages\sklearn\metrics

In [ ]:
# For the inspection tasks (rows) that host the bug, show the distribution of correct labels by
# explanation size and complexity